## 1) Importar bibliotecas

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, QuantileTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import cross_validate, KFold

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np

## 2) Ler base de dados

In [ ]:
dataset = pl.read_parquet(
    source = "../../databases/cleanned_data/diabetes_dataset.parquet"
)

print(dataset.shape)
dataset.head(2)

## 3) Definindo processamentos

### 3.1) Column_Transformer

In [ ]:
standard_scaler = StandardScaler()
columns_standard_scaler = ["age", "bmi", "bp", "s1", "s2", "s5", "s6"]

power_transformer = PowerTransformer(method = "box-cox")
columns_power_transformer = ["s3", "s4"]

one_hot_encoder = OneHotEncoder(
    drop = "if_binary",
)
columns_one_hot_encoder = ["sex"]

preprocessor = ColumnTransformer(
    transformers = [
        ("standard_scaler", standard_scaler, columns_standard_scaler),
        ("power_transformer", power_transformer, columns_power_transformer),
        ("one_hot_encoder", one_hot_encoder, columns_one_hot_encoder)
    ],
    remainder = 'passthrough',
)

### 3.2) transformer

In [ ]:
transformer = QuantileTransformer(
    n_quantiles = 20, output_distribution = "normal"
)

## 4) Treinar modelos

### 4.1) Separar em treino e teste

In [ ]:
X = dataset.drop("target")
y = dataset["target"]

print(X.shape)
print(y.shape)

### 4.2) Função de Dataframe

In [ ]:
def show_validation(
    scores
):

    result_df = []

    for model_name, score in scores.items():

        rows = pl.DataFrame(
                data = score
            ).shape[0]
        
        result_df.append(
            pl.DataFrame(
                data = score
            ).with_columns(
                pl.lit(model_name).alias("model_name"),
                pl.Series(
                    range(rows)
                ).alias("k")
            )
        )

    return pl.concat(result_df)

### 4.3) Função de avaliação

In [ ]:
def train_validade_regression_models(
    X: pl.DataFrame,
    y: pl.DataFrame,
    model,
    preprocessor,
    transformer,
    n_splits: int = 5,
    random_state: int = 1,
    scorings: list[str] = ["r2", "neg_mean_squared_error", "neg_mean_absolute_error"],
):
    
    kf = KFold(
        n_splits = n_splits,
        random_state = random_state,
        shuffle = True
    )
    
    pipeline = Pipeline(
    steps = [
            ("preprocessor", preprocessor),
            ("linear_regression", model)
        ]
    )

    regressor_transformed_target = TransformedTargetRegressor(
    regressor = pipeline,
        transformer = transformer
    )

    scores = cross_validate(
        estimator = regressor_transformed_target,
        X = X,
        y = y,
        cv = kf,
        scoring = scorings
    )

    return scores

### 4.4) Avaliar modelos

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "DummyRegressor": DummyRegressor(strategy = "mean"),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
}

In [ ]:
scores = {
    model_name: train_validade_regression_models(
        X = X,
        y = y,
        preprocessor = preprocessor,
        model = model,
        transformer = transformer,
        n_splits = 10,
    ) for model_name, model in models.items()
}

scores = show_validation(scores)

scores

### 4.5) Função de comparação

In [ ]:
def plot_scores(
    scores: pl.DataFrame,
    columns: list[str],
    column_models: str,
):

    fig, axs = plt.subplots(
        ncols = len(columns),
        figsize = (20, 5),
        gridspec_kw = {
            "wspace": 0.3
        }
    )

    models = scores[column_models].unique().to_numpy()

    axs = axs.flatten()

    for index, ax in enumerate(axs):

        items = [
            scores.filter(
                pl.col(column_models) == model
            )[[columns[index]]].rename({
                columns[index]: f"{model}_{columns[index]}"
            })
            for model in models
        ]

        sns.boxplot(
            data = pl.concat(
                items = items,
                how = "horizontal"
            ),
            ax = ax,
        )

        ax.set_xticks(ticks = np.arange(len(models)), labels = models, rotation = 90)
        ax.set_title(columns[index], fontname = "arial", fontsize = 16)

    plt.show()

In [ ]:
plot_scores(
    scores = scores,
    columns = [
        "fit_time", "score_time", "test_r2", "test_neg_mean_squared_error", "test_neg_mean_absolute_error"
    ],
    column_models = "model_name"
)